# Citations API: Respuestas Verificables con Fuentes

> Aprende a generar respuestas donde cada afirmación viene respaldada
> por una cita exacta del documento fuente, eliminando alucinaciones.

## Objetivos
- Activar citas en documentos de texto plano
- Extraer y mostrar las citas con sus índices de caracteres
- Construir un sistema RAG con atribución automática
- Renderizar respuestas con fuentes verificables

## 1. Instalación de dependencias

In [ ]:
%pip install anthropic --quiet

## 2. Citas básicas en un documento

In [ ]:
import anthropic
import json

client = anthropic.Anthropic()

POLITICA_EMPRESA = """POLÍTICA DE RECURSOS HUMANOS — Versión 5.2 (enero 2025)

ARTÍCULO 1 — HORARIO DE TRABAJO
La jornada laboral ordinaria es de 40 horas semanales distribuidas de lunes a viernes.
El horario flexible permite entrada entre las 8:00 y las 10:00 horas y salida entre las 17:00 y las 19:00.
El teletrabajo está permitido hasta 3 días por semana previo acuerdo con el responsable directo.

ARTÍCULO 2 — VACACIONES
Los empleados tienen derecho a 23 días laborables de vacaciones anuales.
Las vacaciones deben solicitarse con un mínimo de 15 días de antelación a través del portal de RRHH.
No se pueden acumular más de 5 días al año siguiente sin autorización expresa de dirección.

ARTÍCULO 3 — FORMACIÓN
Cada empleado dispone de un presupuesto anual de 1.500€ para formación profesional.
Los cursos deben estar relacionados con el puesto de trabajo o el plan de carrera acordado.
La asistencia a formación durante horario laboral requiere aprobación del responsable con 5 días de antelación.

ARTÍCULO 4 — GASTOS Y DIETAS
Los gastos de desplazamiento por motivos laborales se reembolsan en un plazo máximo de 30 días.
La dieta completa (manutención + alojamiento) para desplazamientos de más de 50km es de 65€/día.
Los gastos superiores a 100€ requieren justificante y aprobación previa del responsable.
"""

def preguntar_con_citas(pregunta: str, documento: str, titulo: str = "Documento") -> dict:
    """Hace una pregunta con citations habilitadas y devuelve respuesta + citas."""
    resp = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=600,
        messages=[{
            "role": "user",
            "content": [
                {
                    "type": "document",
                    "source": {"type": "text", "media_type": "text/plain", "data": documento},
                    "title": titulo,
                    "citations": {"enabled": True}
                },
                {"type": "text", "text": pregunta}
            ]
        }]
    )

    texto = ""
    citas = []
    for bloque in resp.content:
        if bloque.type == "text":
            texto += bloque.text
        elif hasattr(bloque, "citations"):
            for cita in bloque.citations:
                citas.append({
                    "doc_index": cita.document_index,
                    "doc_titulo": getattr(cita, "document_title", titulo),
                    "inicio": getattr(cita, "start_char_index", 0),
                    "fin": getattr(cita, "end_char_index", 0),
                    "texto_citado": getattr(cita, "cited_text", "")
                })
    return {"respuesta": texto, "citas": citas}

preguntas = [
    "¿Cuántos días de vacaciones tengo y con cuánta antelación debo solicitarlos?",
    "¿Cuál es el presupuesto de formación anual?",
    "¿Cuándo necesito aprobación previa para un gasto?",
]

print("=== SISTEMA DE PREGUNTAS CON CITAS ===")
for pregunta in preguntas:
    resultado = preguntar_con_citas(pregunta, POLITICA_EMPRESA, "Política RRHH v5.2")
    print(f"\n❓ {pregunta}")
    print(f"📝 {resultado['respuesta'][:200]}")
    if resultado['citas']:
        print(f"📎 Citas ({len(resultado['citas'])}):")        
        for c in resultado['citas']:
            print(f"   → [{c['doc_titulo']}] chars {c['inicio']}-{c['fin']}: '{c['texto_citado'][:60]}...'")

## 3. Múltiples documentos con atribución por fuente

In [ ]:
CONTRATO_SERVICIO = """
CONTRATO DE SERVICIO — CloudPro SaaS

Cláusula 3.1: El proveedor garantiza una disponibilidad del 99.5% mensual (SLA).
Cláusula 3.2: Las ventanas de mantenimiento programadas serán los domingos entre las 2:00 y las 6:00.
Cláusula 4.1: En caso de incumplimiento del SLA, el cliente recibirá créditos equivalentes al 10% de la cuota mensual por cada hora de caída.
Cláusula 7.2: El contrato se renueva automáticamente por períodos de 12 meses salvo aviso con 60 días de antelación.
"""

FAQ_SOPORTE = """
PREGUNTAS FRECUENTES — Soporte CloudPro

¿Cómo reporto una incidencia? Accede al portal support.cloudpro.com o llama al 900 555 777.
¿Cuál es el tiempo de respuesta? P1 (crítico): 1 hora. P2 (alto): 4 horas. P3 (normal): 24 horas.
¿Puedo cancelar en cualquier momento? Solo con el aviso previo indicado en el contrato.
¿Hay copias de seguridad? Sí, backups diarios con retención de 30 días.
"""

documentos_multi = [
    {
        "type": "document",
        "source": {"type": "text", "media_type": "text/plain", "data": CONTRATO_SERVICIO},
        "title": "Contrato de Servicio",
        "citations": {"enabled": True}
    },
    {
        "type": "document",
        "source": {"type": "text", "media_type": "text/plain", "data": FAQ_SOPORTE},
        "title": "FAQ Soporte",
        "citations": {"enabled": True}
    }
]

resp = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=600,
    messages=[{
        "role": "user",
        "content": documentos_multi + [{
            "type": "text",
            "text": "¿Qué SLA ofrece el servicio y qué compensación recibo si no se cumple? ¿Con cuánto tiempo debo avisar si quiero cancelar?"
        }]
    }]
)

print("=== RESPUESTA CON MÚLTIPLES FUENTES ===")
fuentes_usadas = set()
for bloque in resp.content:
    if bloque.type == "text":
        print("RESPUESTA:", bloque.text)
    elif hasattr(bloque, "citations"):
        for cita in bloque.citations:
            titulo = getattr(cita, "document_title", f"Doc {cita.document_index}")
            fuentes_usadas.add(titulo)
            print(f"\n📎 [{titulo}]: '{getattr(cita, 'cited_text', '')[:80]}...'")

print(f"\nFuentes citadas: {', '.join(fuentes_usadas)}")

## 4. Verificar citas contra el texto original

In [ ]:
def verificar_citas(citas: list, documentos_originales: dict) -> list:
    """Verifica que las citas coincidan exactamente con el texto original."""
    verificaciones = []
    for cita in citas:
        doc_titulo = cita.get("doc_titulo", "")
        texto_original = documentos_originales.get(doc_titulo, "")
        inicio = cita.get("inicio", 0)
        fin = cita.get("fin", 0)

        if texto_original and fin > inicio:
            fragmento_real = texto_original[inicio:fin]
            texto_citado = cita.get("texto_citado", "")
            coincide = fragmento_real.strip() == texto_citado.strip()
            verificaciones.append({
                "doc": doc_titulo,
                "chars": f"{inicio}-{fin}",
                "coincide": coincide,
                "fragmento_real": fragmento_real[:60],
                "texto_citado": texto_citado[:60]
            })
    return verificaciones

# Hacer consulta y verificar citas
resultado_verificado = preguntar_con_citas(
    "¿Puedo teletrabajar? ¿Cuántos días?",
    POLITICA_EMPRESA,
    "Política RRHH v5.2"
)

docs_orig = {"Política RRHH v5.2": POLITICA_EMPRESA}
verificaciones = verificar_citas(resultado_verificado["citas"], docs_orig)

print("=== VERIFICACIÓN DE CITAS ===")
print(f"Respuesta: {resultado_verificado['respuesta'][:200]}")
print(f"\nCitas verificadas: {len(verificaciones)}")
for v in verificaciones:
    icono = "✅" if v["coincide"] else "❌"
    print(f"  {icono} [{v['doc']}] chars {v['chars']}: '{v['fragmento_real']}'")

## 5. Casos de uso y cuándo activar Citations

In [ ]:
import pandas as pd

tabla_uso = pd.DataFrame([
    {"Caso de uso": "Soporte técnico con manual", "Citations": "✅ Sí",
     "Beneficio": "Cliente puede verificar la fuente"},
    {"Caso de uso": "Due diligence legal", "Citations": "✅ Sí",
     "Beneficio": "Trazabilidad de cada afirmación"},
    {"Caso de uso": "Investigación académica", "Citations": "✅ Sí",
     "Beneficio": "Referencias verificables"},
    {"Caso de uso": "RAG empresarial", "Citations": "✅ Sí",
     "Beneficio": "Grounding factual automático"},
    {"Caso de uso": "Chatbot creativo", "Citations": "❌ No",
     "Beneficio": "No aplica, respuesta libre"},
    {"Caso de uso": "Clasificación de textos", "Citations": "❌ No",
     "Beneficio": "No se necesitan fuentes"},
    {"Caso de uso": "Generación de contenido", "Citations": "❌ No",
     "Beneficio": "Creación libre sin restricciones"},
])

print("=== CUÁNDO USAR CITATIONS ===")
print(tabla_uso.to_string(index=False))

print("""
=== RESUMEN TÉCNICO ===

Activar citas:  añadir \"citations\": {\"enabled\": true} al documento
Tipos de cita:
  - char_location: para texto plano (start_char_index, end_char_index)
  - page_location: para PDFs (page_number)

Modelos compatibles: claude-haiku-4-5-20251001, claude-sonnet-4-6
No requiere beta header — disponible en API estándar
""")